Setup Tools

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient (
{
    "travel_server":{
        "transport":"streamable_http",
        "url":"https://mcp.kiwi.com"
    }
}
)
tools = await client.get_tools()

In [4]:
from typing import Dict,Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query:str) -> Dict[str,Any]:
    """Search the query from the web"""
    return tavily_client.search(query)

In [5]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

@tool
def query_playlist_db(query:str) -> str:
    """Query the database for playlist information"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error in querying database: {e}"


Create State

In [6]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

Create Sub Agents

In [7]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model="mimo-v2.5-pro",
    api_key=os.getenv("OPENCODE_API_KEY"),
    base_url="https://opencode.ai/zen/go/v1"
)

travel_agent = create_agent(
    model = model,
    tools = tools,
    system_prompt= """
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the follow
        - Price (lowest, economy class)
        - Duration (shortest)
        - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about
    Once you have found the best options, let the user know your shortlist of options.
    """
)


In [8]:
venue_agent = create_agent(
    model = model,
    tools = [web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following
        - Price (lowest)
        - Capacity (exact match)
        - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    """
)

In [9]:
playlist_agent = create_agent(
    model = model,
    tools = [query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.
    You may need to make multiple queries to iteratively find the best options.
"""
)

Main Coordinator/Orchestrator

In [10]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage,ToolMessage
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    query = f"Find flights from {origin} to {destination}"
    response = await travel_agent.ainvoke(
        {
            "messages": [
                HumanMessage(content = query)
            ]
        }
    )
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find weddin'g venues in {destination} for {capacity} guests"
    response = venue_agent. invoke({"messages": [HumanMessage(content=query) ]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre"""
    return Command[tuple[()]](update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
    )

In [11]:
orchestrator = create_agent(
    model = model,
    tools = [search_flights,search_venues,suggest_playlist,update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. Delegate tasks to your specialists for flights, venues and playlists.
    First find all the information you need to update the state. Once that is done you can delegate the tasks.
    Once you have received their answers, coordinate the perfect wedding for me.
"""
)

Sample Testing

In [12]:
query = f"I am from London and wanted to do destination wedding at Paris for 100 guests, jazz-genre"
response = await orchestrator.ainvoke(
    {
        "messages":[
            HumanMessage(content = query)
        ]
    }
)

In [13]:
print(response['messages'][-1].content)

# 🎉 Your Dream Parisian Wedding Plan 🇫🇷

I've coordinated with all my specialists, and here's your complete destination wedding package from **London to Paris** for **100 guests** with a **jazz** theme!

---

## ✈️ FLIGHTS: London → Paris

| Best Option | Details |
|-------------|---------|
| **Best Value** | easyJet, London Southend → Paris CDG |
| **Date** | 14 September 2026 |
| **Time** | 11:10 → 13:20 (1h 10m flight) |
| **Price** | **€43 per person** |

💡 *Other options available from Luton, Gatwick, Stansted, and Heathrow ranging from €42-49*

---

## 🏛️ VENUE OPTIONS

| Venue | Price | Highlights |
|-------|-------|------------|
| **La Fabrique République** ⭐ | €3,500 – €8,000 | 5.0 rating, perfect for 100 guests |
| **La Maison de Charly** | €4,000 – €6,000 | Full privatization, dancefloor, sound system |
| **Le Georges** (Centre Pompidou) | From €15,000 | Panoramic Paris views, modern design |
| **The Peninsula Paris** | From €18,750 | Eiffel Tower view terrace, luxury |
| **